In [ ]:
%load_ext autoreload
%autoreload 2

import os
import numpy as np
import torch
import matplotlib.pyplot as plt
from ipywidgets import interact, IntSlider

# 2. Recurrent Neural Networks & Transformers
### University of Cambridge
### Engineering Part IIB/Module 4F12: Computer Vision
### Deep Learning for Computer Vision

*Lecturer: Matthew Johnson*

**Setup:** Before running this notebook, ensure you have installed the dependencies and downloaded the pre-computed results:
```bash
pip install -r requirements.txt
python download_results.py
```

## Recurrent Neural Networks (RNNs)

So far we have looked at data which is static, i.e. it does not change over time. However, many tasks involve ordered sequences of data, such as text, audio, or video. Recurrent Neural Networks (RNNs) are a type of neural net which can process sequences of data by maintaining a hidden state that is updated at each time step. The idea has been around since the very beginning of the field, with even Frank Rosenblatt exploring recurrent structures [Rosenblatt, 1960]. Here is a simple RNN architecture, called an Elman network [Elman, 1990]:

![RNN](images/rnn.svg)

The hidden state $\mathbf{h}$ is updated at each time step by taking the previous hidden state and the current input $\mathbf{x}$ and applying a linear transformation to each, adding the result, and then applying a non-linear activation function. The hidden state is used by the network to contain information about the sequence so far. The output $\mathbf{y}$ is then computed from the hidden state. Note that there is an additional output transform `h2o` that produces an $\mathbf{o}$ tensor at each step.

An RNN is implemented using a loop, following a pattern like the one below:

```text
h_0 = 0
for t = 1 to T:
    h_t = sigma(W_hh @ h_{t-1} + W_xh @ x_t)
    o_t = sigma(W_ho @ h_{t-1})
y = h_T
```

Crucially, the same fully-connected layers get used in each iteration. The only elements that are changing are the input $\mathbf{x}$ and the hidden state $\mathbf{h}$. This allows the network to process sequences of arbitrary length, essentially creating a dynamically deep neural network (with shared weights at each level). This strength of RNNs, however, is also their weakness. The network has to learn to maintain information over long sequences, which can be difficult.

### Vanishing Gradients in RNNs

As sequence length increases, a form of the vanishing gradient problem returns. The network receives a signal from the loss function at the end of the sequence, but that signal gets diluted as it is applied over the full sequence. This can result in either slow or non-convergent training:

In [ ]:
import rnn
rnn.vanishing_gradient("backpropagation")

### Self-Supervision

The sequence itself holds a solution to this problem: causality. In an ordered sequence, for example a word or a sentence, the next item is dependent on the ones which came before. As such, we can set up an auxiliary task in which we predict $\mathbf{x}_{t+1}$ using $\mathbf{o}_{t}$.

$$
\mathcal{L}_{\text{recon}} = \sum_{t=1}^{T} H(\mathbf{x}_{t+1}, P(c~|~\mathbf{x}_t))
$$

$$
P(c~|~\mathbf{x}_t) = s(\mathbf{o}_t)
$$

This is a form of *self-supervision*, in which a neural network can be trained in a supervised manner but without requiring the production of labelled data by a human annotator. As it can be applied at each step, it provides a powerful auxiliary loss which can help the network learn to maintain information over long sequences.

### Sequential Datasets

In order for an RNN to be able to operate over a dataset of ordered sequences, the sequences first need to be *tokenised*. This means that the sequence is broken up into tokens, typically represented by an integer, which are then fed into the network. For text, this is typically done either at the word level or the character level. In the case of words, a dictionary is constructed that maps each word in the vocabulary to either an integer or a special "unknown" token. For characters, each word is broken into a sequence of characters and each character is mapped to an integer. Finally, before being passed into the network, this integer gets converted into a one-hot vector.

The first sequential dataset we are going to examine is one which contrasts 6-letter words in English, French (without diacritics), and Japanese words which, when written using the same Latin alphabet (romaji using the Hepburn system), also have 6 letters. It is balanced, with 8,391 words per language:

$$
\begin{matrix}
\texttt{reigai} & \texttt{mousey} & \texttt{trempa} \\
\texttt{hiyoko} & \texttt{vexing} & \texttt{boutez} \\
\texttt{fuurin} & \texttt{quilts} & \texttt{ivraie} \\
\texttt{susumo} & \texttt{memory} & \texttt{masure} \\
\texttt{genzon} & \texttt{roofer} & \texttt{baguai} \\
\texttt{rizumu} & \texttt{awless} & \texttt{mirera} \\
\texttt{itonan} & \texttt{sadden} & \texttt{crions} \\
\end{matrix}
$$

We will train an RNN to predict whether a word is either English, Japanese, or French, and also to predict the next character in each sequence.

### RNN on Language Data

If we train an RNN on this word dataset with a trinary classification task (Japanese/English/French) and a semi-supervised task of predicting the next character, we get the following results. We see four groups: certain/right, uncertain/right, uncertain/wrong, and certain/wrong. It is good practice to look at results in this way, as it helps one understand the kinds of mistakes the model is making. Also, due to how the RNN works, we can predict a class at any point in the sequence, not just at the end:

In [ ]:
from ipywidgets import Play, HBox, jslink, interactive_output

rnn_lang_results = rnn.load_language_rnn()
rnn.show_language_rnn(rnn_lang_results)

rnn_dataset, rnn_frames = rnn.language_rnn_slider_data(rnn_lang_results, num_examples=20)

plt.rc('font', size=15)
def slide_rnn(step):
    fig = plt.figure(figsize=(9, 4))
    rnn.plot_rnn_frame(fig, rnn_dataset, rnn_frames[step])
    plt.show()

play = Play(value=0, min=0, max=len(rnn_frames)-1, step=1, interval=500)
slider = IntSlider(value=0, min=0, max=len(rnn_frames)-1)
jslink((play, 'value'), (slider, 'value'))
out = interactive_output(slide_rnn, {'step': slider})
display(HBox([play, slider]), out)

### Images as Sequences

This is a course on Computer Vision, so it may seem odd that we are focusing on a technology that is designed for 1-dimensional sequences. However, if we break the image into parts, e.g. into a grid of patches, then we can walk over the image in a sequence from left to right, top to bottom.

![RNN Patch](images/rnn_patch.svg)

We will use another dataset for this task, a slightly expanded version of MNIST which adds the 26 letters of the Latin alphabet to the 10 digits to create 36 classes. We will break each $28 \times 28$ image into a $7 \times 7$ grid of $4 \times 4$ patches, resulting in a sequence length of 49.

In [ ]:
rnn.image_to_sequence()

### Patch Tokenisation

We have glossed over an important point, however. The letters in a word are natural tokens, in that they have a bijection into a subset of the natural numbers (e.g. $a = 0$, $b = 1$, etc.). However, each patch in our 49-patch sequence is 16 pixels, and thus belongs to a set of $2^{24}$ possible patches, which is a much larger dictionary than we ideally want to use. There are a couple options at our disposal:

- **Learned Embedding**: Create a mapping from the patch pixels to a fixed-length input vector. This mapping can be anything from a simple linear transform to a full convolutional neural net. While it can be learned during training it is often helpful for it to be pretrained on a large corpus of images. The resulting vector can be hard to interpret unless the mapping is invertible (e.g. as is the case with an auto-encoder).
- **Dictionary Encoding**: Create a set of statistical models of patches and assign each patch to the model which best fits its appearance (e.g. via clustering). If a patch falls too far out of the range of the models, it can be assigned a special "unknown" token. For small, simple patches this can be quite effective, and the statistical models provide a clear context and meaning for the tokens.

As we want to have the RNN learn to understand images, we will use the learned embedding method.

To train an RNN on EMNIST, the classification loss remains the same (e.g. cross entropy over the 36 classes) but the self-supervision loss will be different. The network is learning an embedding, so we must have some way of evaluating that embedding, which in turn means we need to have some way of inverting the embedding to produce a patch. As the patches in this case are $4 \times 4$ grayscale images, we can perform a linear transform from the hidden state to produce a 16-element vector, and reshape that vector to be a $4 \times 4$ image.

We can treat the resulting $4 \times 4$ image of logits as a *Bernoulli patch distribution*. A Bernoulli distribution models the likelihood of binary outcomes:

$$
P(x) = \begin{cases} p & \text{if } x = 1 \\ 1 - p & \text{if } x = 0 \end{cases}
$$

When combined in a $4 \times 4$ grid, we can model the probability that a pixel is on or off by passing the logits through a sigmoid function. Here is an example of a Bernoulli patch distribution, and some samples from that distribution:

In [ ]:
rnn.bernoulli_samples()

We can train this distribution using binary cross-entropy loss:

$$
\mathcal{L}_{\text{recon}} = -\sum_{i=1}^{16} \left(x_i \log(P(i)) + (1 - x_i) \log(1 - P(i))\right)
$$

There will be more than one likely patch appearance in most cases, however, so we want to increase the expressivity of this model by making it a *mixture* of Bernoulli distributions. The network outputs $N$ patch logits $\mathbf{o}_j$ and $N$ mixture values $\mathbf{z}$, such that $P(i)$ is computed as:

$$
\begin{aligned}
P(i) &= \sum_{j}^{N} P(i|z_j)P(z_j) \\
P(z_j) &= \frac{\exp(z_j)}{\sum_{k}^{N} \exp(z_k)} \\
P(i|z_j) &= \frac{\exp(o_{ji})}{1 + \exp(o_{ji})}
\end{aligned}
$$

### RNN on EMNIST

With all of this in place, we can train our RNN on EMNIST. As before, we can predict the class from partial images. Given the nature of the task and the similarity between various classes, it is not surprising that the network needs to see most of the image before it makes an accurate prediction:

In [ ]:
emnist_rnn_results = rnn.load_emnist_rnn()
rnn.show_emnist_rnn(emnist_rnn_results)

### Transfer Learning / Fine-tuning

What is really exciting about RNNs, however, is that we can take this network which has been trained on one classification task, and fine-tune it to work on another. In this case, we will take the network which has been trained to classify the 36 classes of the EMNIST dataset, and fine-tune it to classify whether the image is a digit or a letter. The way we fine-tune it is by replacing just the classification portion of the network, but leaving the rest of the weights frozen. This results in fewer weights to train, and surprisingly good performance:

In [ ]:
emnist_ft_results = rnn.load_emnist_rnn_finetune()
rnn.show_emnist_rnn_finetune(emnist_ft_results)

Because of the semi-supervised auxiliary task of reconstructing the image, the network has learned a rich-enough representation for its hidden state that it can also be used for this entirely new task. Note that a network trained from scratch on the dataset achieves the same accuracy (around 92%).

### LSTM

RNNs are incredibly powerful. However, they have some problems. First, our solution for the vanishing gradient problem is not perfect. The network is able to learn connections between tokens which are nearby in the sequence, but longer-range connections are more difficult. One way of addressing this is via Long Short-Term Memory (LSTM) [Hochreiter & Schmidhuber, 1997]:

![LSTM](images/lstm.svg)

$$
\begin{aligned}
\mathbf{z}_t &= \begin{bmatrix} \mathbf{x}_t & \mathbf{h}_{t-1} \end{bmatrix} & \mathbf{i}_t &= \tanh(\mathbf{W}_{ZI} \mathbf{z}_t + \mathbf{b}_{ZI}) \\
\mathbf{f}_t &= \sigma(\mathbf{W}_{ZF} \mathbf{z}_t + \mathbf{b}_{ZF}) & \mathbf{o}_t &= \sigma(\mathbf{W}_{ZO} \mathbf{z}_t + \mathbf{b}_{ZO}) \\
\mathbf{c}_t &= \sigma(\mathbf{W}_{ZC} \mathbf{z}_t + \mathbf{b}_{ZC}) & \mathbf{h}_t &= \mathbf{o}_t \circ \tanh(\mathbf{m}_t) \\
\mathbf{m}_t &= \mathbf{f}_t \circ \mathbf{m}_{t-1} + \mathbf{i}_t \circ \mathbf{c}_t
\end{aligned}
$$

### RNN Limitations

This architecture separates out its internal state into *cell state* $\mathbf{m}$ and *hidden state* $\mathbf{h}$. $\mathbf{m}$ acts as the long-term memory and persists throughout the sequence. The short-term memory $\mathbf{h}$ functions like the hidden state of a normal RNN and contains information biased towards more recent tokens.

The innovation of LSTMs is in how they update $\mathbf{m}$. At each time step, the network decides what to forget ($\mathbf{f}$), what to update ($\mathbf{i}$), and what to add ($\mathbf{c}$). This allows the network to learn to perform more complex tasks that involve longer-range dependencies.

While the LSTM addresses some of the issues with long-term connections, the core problem of RNNs remains: computational cost. RNNs are slow to train and slow to run, as they require unrolling over the entire sequence. Training and evaluation of a normal neural net scales with the number of weights ($O(N)$), but an RNN scales with the number of weights and the number of time steps ($O(NT)$). What we need is a way to process a sequence all at once, but without sacrificing the context provided by causality.

## Transformers

### Positional Encoding

Let us return to a simple Multi-layer Perceptron, like the ones we worked with in the beginning of the course. We have primarily focused on classification, but you can also use an MLP to perform *regression*, in which the MLP produces a continuous output. When used this way, neural nets become a powerful tool for function approximation.

Let us see what happens when we train an MLP to predict the output of $y = 2 + \sin(x\pi) + \frac{1}{2}\sin(2x\pi) - \frac{1}{5}\cos(5x\pi)$ from 32 points in the range $[0, 2]$. Note how the result improves with positional encoding, which augments the input with a combination of sines and cosines:

$$
\begin{aligned}
\mathbf{x}_{pos} &= \begin{bmatrix} c_1 & s_1 & \ldots & c_i & s_i & \ldots & c_D & s_D \end{bmatrix} \\
c_i &= \frac{1}{i}\cos(i\pi x) \\
s_i &= \frac{1}{i}\sin(i\pi x)
\end{aligned}
$$

This allows the network to learn to model the function more effectively [Tancik et al., 2020]:

In [ ]:
import transformers_model
transformers_model.periodic(num_layers=1, hidden_size=64, position_size=0)
transformers_model.periodic(num_layers=2, hidden_size=256, position_size=0)
transformers_model.periodic(num_layers=1, hidden_size=64, position_size=16)

### 2D Positional Encoding

The same phenomenon is even more pronounced if the function we are trying to approximate is an image. In the centre, we see the result with a standard MLP. On the right we see the result of using positional encoding. The formula has slightly changed:

$$
\begin{aligned}
\mathbf{x}_{pos} &= \begin{bmatrix} c_0 & s_0 & \ldots & c_i & s_i & \ldots & c_{D-1} & s_{D-1} \end{bmatrix} \\
c_i &= \begin{bmatrix} \cos(f_i \pi x) & \cos(f_i \pi y) \end{bmatrix} \\
s_i &= \begin{bmatrix} \sin(f_i \pi x) & \sin(f_i \pi y) \end{bmatrix} \\
f_i &= 2^{\frac{i\sigma}{D}} - 1
\end{aligned}
$$

where $2^\sigma - 1$ is the maximum desired frequency. Using positional encoding we can process an entire sequence (or image) of tokens at once, rather than having to process them one at a time, as the encoding will provide the network with the needed context:

In [ ]:
transformers_model.cat_image(position_size=16)

### Attention Mechanisms

Processing a whole sequence in one go addresses the issue of computational complexity, but it does not solve the problem of long-range dependencies. For this, we need a new kind of network architecture, one which can learn to focus on different parts of the input at different times. This is the idea behind *attention mechanisms*.

![SDPA](images/sdpa.svg)

The form of attention shown here is known as *scaled dot-product attention*. The network performs a linear mapping of each token into three different $T \times D$ matrices: a query $Q$, a key $K$, and a value matrix $V$. The rows of $Q$ indicate what the network is looking for, the rows of $K$ encode the input token for lookup, and the rows of $V$ are output tokens. $Q$ and $K$ are combined to create a $T \times T$ attention matrix $A$:

$$
A = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)
$$

where $d_k$ is the length of the key vector. Using $\sqrt{d_k}$ as a normalising factor ensures the network does not over-focus on a subset of tokens.

$A$ is then used to perform a weighted sum of the rows of $V$:

$$
O = AV
$$

The rows of the resulting $T \times D$ matrix $O$ are the new tokens. We can produce multiple $(Q, K, V)$ triplets from the input tokens, which is referred to as having multiple "heads". This allows the network to learn to attend to different parts of the input at the same time:

![MHSA](images/mhsa.svg)

### Vision Transformer (ViT)

The combination of positional encoding and multi-head self attention results in one of the most widely used architectures in modern machine learning: the *transformer*:

![ViT](images/vit.svg)

The transformer architecture is quite similar to ResNet, in that it uses residual links to increase gradient flow between identical blocks. However, instead of using convolutional layers, it uses multi-head self attention. The output of the attention layer is then passed through a feed-forward neural network, and the result is added back to the input. This is done multiple times, and the final tokens are passed through a linear layer to produce the final output.

The architecture above is an encoder-only transformer, which in the context of computer vision is called a Vision Transformer or ViT [Dosovitskiy et al., 2021] and can be used as a drop-in replacement for a CNN. A single special token, called the "class" token, is prepended to the sequence and used to predict the class label. Due to attention, it can look at all the other tokens in order to make a decision.

One change from ResNet is the use of *layer normalisation* [Ba et al., 2016] at regular intervals to stabilise training. Layer normalisation is similar to batch normalisation, but it computes the mean and variance over the channels for each activation $\mathbf{y}$, instead of over the batch for each feature:

$$
\begin{aligned}
\mu_d &= \frac{1}{N}\sum_i y_d^i & \sigma_d &= \sqrt{\frac{1}{N}\left(\sum_i y_d^i - \mu_d\right)^2} \\
\hat{\mathbf{y}_d} &= \frac{\mathbf{y}_d - \mu_d}{\sigma_d + \epsilon}
\end{aligned}
$$

Transformers also use a specialised non-linearity called a Gaussian Error Linear Unit (GELU) [Hendrycks & Gimpel, 2016], which weights an input $x$ by how much greater it is than other inputs. Based on the assumption that $x$ is normally distributed, this is equivalent to multiplying $x$ by the cumulative distribution of $\mathcal{N}$:

$$
\text{GELU}(x) = x P(X \leq x) = x \Phi(x) = x \frac{1}{2}\left[1 + \text{erf}\left(\frac{x}{\sqrt{2}}\right)\right]
$$

In [ ]:
import activations
x = np.arange(-50, 50) / 10.0
plt.figure(figsize=(4, 4))
plt.ylim(-1, 5)
plt.plot(x, activations.relu(x), label="ReLU")
plt.plot(x, activations.gelu(x), label="GELU")
plt.legend()
plt.tight_layout()
plt.show()

### Transformer on EMNIST

Let us return to the EMNIST dataset and see how ViT performs. As we can see, it exceeds the performance of the patch RNN on the same task. We can use the attention map of the classification token to get a sense of where the transformer looks in order to make its decision for each block:

In [ ]:
from ipywidgets import Play, HBox, jslink, interactive_output

emnist_vit_results = transformers_model.load_emnist_vit()
transformers_model.show_emnist_vit(emnist_vit_results)

vit_frames = transformers_model.vit_attention_slider_data(emnist_vit_results)

plt.rc('font', size=15)
def slide_vit(step):
    fig = plt.figure(figsize=(9, 4))
    transformers_model.plot_vit_attention_frame(fig, vit_frames[step])
    plt.show()

play = Play(value=0, min=0, max=len(vit_frames)-1, step=1, interval=500)
slider = IntSlider(value=0, min=0, max=len(vit_frames)-1)
jslink((play, 'value'), (slider, 'value'))
out = interactive_output(slide_vit, {'step': slider})
display(HBox([play, slider]), out)

### Transformer on Language

Here we see results on the language dataset. As can be seen, the transformer is able to perform the same sequence-based learning tasks as RNNs, but with increased efficiency:

In [ ]:
from ipywidgets import Play, HBox, jslink, interactive_output

lang_vit_results = transformers_model.load_language_vit()
transformers_model.show_language_vit(lang_vit_results)

lang_ft_results = transformers_model.load_language_gpt_finetune()
lang_dataset, lang_frames = transformers_model.language_transformer_slider_data(lang_ft_results)

plt.rc('font', size=15)
def slide_language(step):
    fig = plt.figure(figsize=(9, 4))
    transformers_model.plot_language_transformer_frame(fig, lang_dataset, lang_frames[step])
    plt.show()

play = Play(value=0, min=0, max=len(lang_frames)-1, step=1, interval=500)
slider = IntSlider(value=0, min=0, max=len(lang_frames)-1)
jslink((play, 'value'), (slider, 'value'))
out = interactive_output(slide_language, {'step': slider})
display(HBox([play, slider]), out)

### Comparison with CNNs

How does the ViT compare with CNNs on CIFAR-10? As we can see, when trained in the same way as the FCN we examined earlier it performs considerably poorly. It is not only worse, but uses an order of magnitude more parameters. The ViT trained here had 6 layers, 4 heads, and a token size of 128, for a grand total of 1,197,322 trainable weights, in contrast with 88,938 for the FCN.

With extensive data augmentation, ViT can just about match the FCN. The line at 98.5%, however, is the result of fine-tuning a ViT that was pretrained on ImageNet-21k (which contains 14,197,122 images in 21,841 classes). This is what transformers provide: efficient use of increased capacity. Up until now we have wanted to reduce model capacity, but only due to a lack of data and compute budget for training. As we will see, given sufficient compute transformers can be trained in such a way as to make use of so much data that we can effectively use their increased capacity without overfitting:

In [ ]:
cifar_vit_results = transformers_model.load_cifar_vit()
transformers_model.show_cifar_vit(cifar_vit_results)

### Decoder Transformers

There is one difficulty which we encounter, however, by leaving the RNN architecture behind. Left to its own devices, the transformer can (and will) look at every token when predicting the next token in the sequence, including the next token itself. As such, we need to introduce a masking mechanism to prevent the network from looking at tokens that have not been generated yet.

![Decoder Transformer](images/decoder_transformer.svg)

This architecture is called a *decoder-only transformer*, and is designed for use in generative pretraining. The masking happens at the level of the attention matrix $A$. The upper triangular portion (not including the diagonal) is set to $-\infty$ before the softmax is applied, which has the effect of setting the probability of those tokens to zero. This is called *masked self-attention*, and prevents tokens from "looking ahead" to see tokens that come later in the stream. To ensure we still learn about the starting token, we prepend a special "start" token to the sequence, so that the actual sequence starts at the second token.

### Generative Pretraining

What is so exciting about generative pretraining is that a network trained in this way can then be fine-tuned for use on a different task. By concatenating or taking the mean of the output tokens and then passing them into a classifier, we can train the network to perform a classification task. Here is the result of training a decoder transformer on the language dataset and then fine-tuning with language labels for classification:

In [ ]:
lang_gpt_results = transformers_model.load_language_gpt()
lang_gpt_ft_results = transformers_model.load_language_gpt_finetune()
transformers_model.show_language_gpt(lang_gpt_results, lang_gpt_ft_results)

Another exciting aspect of generative pretraining is that we can sample from the trained model to create new data. This can often give us insight into what the model has learned. We gather samples by starting from a special "start" token and iteratively sampling from $P(t|\mathbf{t})$ until we reach the end of the sequence.

### Patch Tokenisation Redux

Learned patch embeddings work quite well for encoder-only transformers, but they are not as useful for decoder-only transformers, which benefit from the quantization that a dictionary provides. As such, we will use the *K-means clustering algorithm* to group similar patches together to create a dictionary. K-means clustering works as follows:

```text
procedure KMeans(X, K, T):
    M = {mu_0, mu_1, ..., mu_K}
    z = sample(K, |X|)
    delta = infinity
    t = 0
    while delta > 0 and t < T:
        for k in K:
            I_k = {i | z_i = k}
            if |I_k| > 0:
                mu_k = mean(x_i for i in I_k)
            else:
                mu_k = random(K)
        delta = 0
        for i in N:
            k_new = argmin_k ||x_i - mu_k||
            if k_new != z_i:
                delta += 1
            z_i = k_new
        t += 1
    return M, z
```

If we randomly select 200,000 patches from the training set and cluster them into 128 clusters, we get a patch dictionary. Once we have the dictionary, we must go through all our images and tokenise them:

In [ ]:
import patch_clustering
emnist_p, emnist_gt = patch_clustering.emnist_patches()
patch_clustering.analyze(emnist_p, emnist_gt)
patch_clustering.figures(emnist_p, emnist_gt)

### Decoder Transformer on EMNIST

Here is an example of pretraining on the EMNIST dataset and then fine-tuning it to perform classification, along with samples from the model generated using our new patch dictionary. The power of generative pretraining is that it allows you to train a network on a large, unsupervised dataset, and then fine-tune it on a smaller, supervised dataset. This is a large part of why the GPT architecture has been so successful:

In [ ]:
emnist_gpt_results = transformers_model.load_emnist_gpt()
emnist_gpt_ft_results = transformers_model.load_emnist_gpt_finetune()
transformers_model.show_emnist_gpt(emnist_gpt_results, emnist_gpt_ft_results)

### Sampling Strategies

Sampling from generative models is an entire lecture series unto itself, but we can look at some common strategies used to sample from models like GPT. The generative model produces a distribution $P(t|\mathbf{t})$. We can sample from this distribution to construct the token sequence iteratively:

$$
\mathbf{t}_{i+1} = [\mathbf{t}_{i} ~ t] ~|~ t \sim P(t~|~\mathbf{t_i})
$$

where $\mathbf{t}_0 = [t_\alpha]$ and $t_\alpha$ is the start token. If we want to bias things slightly more towards high-probability transitions, we can use a *temperature* value $\tau$ to scale the logits $y$ before sampling as $y_\tau = \frac{y}{\tau}$. A temperature of 1 is the same as sampling directly from the model distribution, but as the temperature approaches 0 the distribution entropy decreases and sampling becomes more greedy:

In [ ]:
transformers_model.temperature_diagram()

It is often the case in token prediction that the token set is very large and $P(t|\mathbf{t_s}) \approx 0$ for most tokens. As such, we often want to restrict sampling to the most likely tokens:

- **Top-K sampling**: Set all but the top $K$ values to 0 and renormalise before sampling.
- **Top-P sampling**: Find the minimum set of tokens which achieves a probability threshold $p_{\text{thresh}}$ and renormalise before sampling.
- **Beam search**: A heuristic search algorithm that explores the most likely sequences by expanding the most promising children of a limited set of nodes (the beam):

```text
procedure BeamSearch(T, P, K):
    B = {(0, [start])}
    for i = 1 to N:
        B_new = {}
        for (s, t) in B:
            for token in T:
                t_new = [t, token]
                s_new = s + log P(token | t)
                B_new = B_new + {(s_new, t_new)}
        B = top-K(B_new, K)
    return argmax_{t in B} P(t)
```

### Encoder-Decoder Transformer

Now that we have looked at encoder-only transformers and decoder-only transformers, we will look at the full transformer architecture [Vaswani et al., 2017]:

![Transformer](images/transformer.svg)

It was originally developed for use in machine translation. The input sequence is passed through the encoder. Simultaneously, the output sequence is passed through the decoder (with masking to prevent lookahead). Then, each token in the decoder is able to look at the encoded input to produce its prediction of the next token. In order to explore this architecture, we are going to develop a *Vision and Language Model* or VLM, using a new dataset.

### Vision-Language Model: Synthetic Postcodes

We will use this opportunity to look at one of the most powerful techniques in machine learning: synthetic data. We are going to build a VLM that can read handwritten UK postal codes. It would be laborious to build the dataset needed from scratch and to ensure it had the needed variation for robustness. Instead, we can generate a random postal code and then use EMNIST to construct a synthetic image which matches it.

There are 11,225 sector prefixes and 400 valid unit codes and thus almost 4.5 million valid postal codes in the UK, but the dataset we have created will focus on 78,575 of them: 67,350 for training and 11,225 for validation. Unlike other sequence datasets we have explored, this one includes a special token for spaces, as postal codes are often written with spaces between the two halves and can be of varying lengths:

In [ ]:
from ipywidgets import Play, HBox, jslink, interactive_output

postcodes_results = transformers_model.load_postcodes()
transformers_model.show_postcodes(postcodes_results)

postcode_frames = transformers_model.postcodes_slider_data(postcodes_results)

plt.rc('font', size=15)
def slide_postcodes(step):
    fig = plt.figure(figsize=(10, 5))
    transformers_model.plot_postcodes_frame(fig, postcode_frames[step])
    plt.show()

play = Play(value=0, min=0, max=len(postcode_frames)-1, step=1, interval=500)
slider = IntSlider(value=0, min=0, max=len(postcode_frames)-1)
jslink((play, 'value'), (slider, 'value'))
out = interactive_output(slide_postcodes, {'step': slider})
display(HBox([play, slider]), out)

## State of the Art

### VGGT

Traditionally, recovering 3D structure from images has relied on multi-stage pipelines involving feature matching, triangulation, and iterative optimisation such as Bundle Adjustment. The Visual Geometry Grounded Transformer (VGGT) [Wang et al., 2025] replaces this entire pipeline with a single feed-forward transformer that, given one or more images of a scene, directly predicts all key 3D attributes in a single forward pass:

![VGGT Overview](images/vggt_overview.png)
*Reproduced from https://github.com/facebookresearch/vggt*

The outputs include camera extrinsics and intrinsics, per-pixel depth maps, dense 3D point maps, and point tracking features. Input images are first patchified into tokens using a pretrained DINOv2 backbone. Special camera tokens are appended for each frame, and the combined token sequence is processed through 24 layers of *alternating attention*: frame-wise self-attention (within each image) alternates with global self-attention (across all images).

Prediction heads decode the transformer's output tokens into the final 3D quantities. A lightweight camera head (four self-attention layers plus a linear layer) predicts camera parameters from the camera tokens. Dense outputs — depth maps, point maps, and tracking features — are produced from the image tokens via a Dense Prediction Transformer (DPT) head:

![Dense Prediction Transformer](images/dense_prediction_transformer.png)
*Reproduced from https://github.com/CV-IP/DPT-1*

A key insight is that the predicted quantities are over-complete: depth maps and camera parameters together determine the point maps. However, jointly supervising all outputs during training improves accuracy across all tasks. Remarkably, despite having minimal 3D-specific inductive biases, VGGT achieves state-of-the-art results on camera pose estimation, multi-view depth estimation, dense point cloud reconstruction, and point tracking — all in under one second.

### CLIP

Contrastive Language-Image Pre-training (CLIP) is a state of the art VLM which is able to produce images from text prompts and vice versa. It begins with contrastive pre-training:

![CLIP Overview](images/clip_overview_0.png)
*Reproduced from https://openai.com/index/clip/*

The network consists of two encoders: a ViT for images, and an encoder-only transformer for text. For each training batch of image and text pairs, CLIP computes the cosine similarity between all pairs of encodings. The loss function maximises the similarity between positive pairs and minimises the similarity between negative pairs.

After pretraining, CLIP can be used as a *zero-shot classifier*, which means that it can be used for tasks it was not explicitly trained for without any fine-tuning. For example, if you wanted to build a classifier you can construct sentences like "a photo of a dog" or "a drawing of a cat" and pass them through the text encoder. Then, you can pass the image through the image encoder and compute the cosine similarity between the two encodings. The class with the highest similarity is the predicted class:

![CLIP Zero-Shot](images/clip_overview_1.png)
*Reproduced from https://openai.com/index/clip/*

In [ ]:
import clip
clip.demo()

We can also examine the similarity between images and text directly:

In [ ]:
clip.image_text_similarity(
    image_paths=["data/cat.jpg", "data/panda.jpg"],
    texts=["a photo of a panda", "a photo of a cat", "a photo of a city", "a photo of nature"]
)

### DINOv2

Knowledge **DI**stillation with **NO** labels, or DINO [Oquab et al., 2024], is a family of self-supervised Vision Transformer models that learn general-purpose visual features from images alone, without any text supervision or labels. Unlike CLIP, which requires paired image-text data, DINO is trained purely on images using a *discriminative self-supervised* objective. The key idea is a student-teacher framework: a student network learns to match the outputs of a teacher network, which is maintained as an exponential moving average (EMA) of the student's weights. The largest model (ViT-g, 1.1B parameters) is trained from scratch, and smaller models are then *distilled* from it:

![DINOv2 ViT-L/14](images/dinov2_vitl14.svg)

The training loss combines two self-supervised objectives. The first is the *DINO loss*, an image-level objective. Both student and teacher receive different crops of the same image. Their class tokens are passed through separate MLP heads to produce probability distributions $p_s$ and $p_t$ over a set of prototypes. The loss is the cross-entropy between them:

$$
\mathcal{L}_{\text{DINO}} = - \sum p_t \log p_s
$$

![DINO Loss](images/dino_loss.png)

Note how the student receives distorted crops at random locations, while the teacher receives only the full image. This makes the network learn how to create a robust and stable representation.

The second is the *iBOT loss* [Zhou et al., 2022], a patch-level objective inspired by masked image modelling. Some input patches are randomly masked in the student but remain visible to the teacher. The student's iBOT head predicts the teacher's patch-level outputs for the masked positions:

$$
\mathcal{L}_{\text{iBOT}} = - \sum_i p_t^i \log p_s^i
$$

where $i$ indexes the masked patches. This term is critical for dense prediction tasks such as segmentation and depth estimation, improving performance by almost 3%.

![iBOT Loss](images/ibot_loss.png)

Let us visualise the augmentations used during DINO training:

In [ ]:
import dino
dino.dino_augmentations()

And the masking used for the iBOT loss:

In [ ]:
dino.dino_masking()

### DINO Features

The resulting features work "out of the box" across a wide range of tasks — classification, segmentation, depth estimation, and image retrieval — using only frozen features with a simple linear classifier, closing the gap with weakly-supervised models like CLIP. Remarkably, properties such as semantic part understanding emerge without explicit supervision:

In [ ]:
dino.dino_features()

### Responsible AI

We have come full circle. At the beginning of this module, we looked at this image:

![Train with captions](images/train_with_captions.jpg)

These image captions were created using a state-of-the-art VLM. Large model training is beginning to raise ethical questions. The datasets are so large that training requires resources beyond the reach of all but nation states and the largest corporations. For example, CLIP is trained on 400 million image-text pairs, requiring 30 days of training across 592 V100 GPUs. For context, this would cost \$1,000,000 to train on AWS.

Even the power requirements raise serious questions about the environmental impact of training these models. However, we know that vision systems exist which do not have these running costs. The human brain can achieve incredible feats of vision with only 12 watts of power. So, while we have accomplished much, in some ways, our journey towards artificial intelligence is just beginning.